In [ ]:
# Cell 1: INSTALL + IMPORTS
import cv2
import urllib.request
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import subprocess
import time
import threading
print("✅ Alle imports geladen!")

In [11]:
# ============================================
# FATIGUE DETECTOR: 1 GEZICHT + MOND + COUNTS
# ENKEL ALARM BIJ SLAPEN
# ============================================

import cv2
import urllib.request
import os
import numpy as np
import torch
import torch.nn as nn
import time
import threading
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

try:
    import winsound
except:
    winsound = None

print("Imports geladen!")

# ============================================
# DEVICE
# ============================================
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print('🚀 MPS!')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print('🚀 CUDA!')
else:
    device = torch.device('cpu')
    print('CPU')

# ============================================
# MODELS
# ============================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64*28*28, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.model(x)

eyes_model = SimpleCNN().to(device)
eyes_model.load_state_dict(torch.load(
    '/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/Model/eyes/fatigue_model.pth',
    map_location=device
))
eyes_model.eval()
print('✅ Eyes OK')

yawn_model = models.efficientnet_b0(weights='DEFAULT')
yawn_model.classifier[1] = nn.Linear(yawn_model.classifier[1].in_features, 2)
yawn_model.load_state_dict(torch.load(
    '/Users/nachatissa/Desktop/SCHOOL/SEM2/Advanced AI/untitled folder/Advanced-AI/Model/yawn/yawn_model.pth',
    map_location=device
))
yawn_model.to(device)
yawn_model.eval()
print('✅ Yawn OK')

# ============================================
# TRANSFORMS
# ============================================
eyes_transform = transforms.Compose([transforms.ToTensor()])
yawn_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ============================================
# FACE CASCADE
# ============================================
face_cascade_url = 'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml'
face_path = 'haarcascade_frontalface_default.xml'

if not os.path.exists(face_path):
    try:
        urllib.request.urlretrieve(face_cascade_url, face_path)
        print('✅ Face cascade gedownload')
    except:
        print('⚠️ Face cascade download gefaald')

face_cascade = cv2.CascadeClassifier(face_path)
if face_cascade.empty():
    print('❌ Face cascade niet geladen!')
    exit()

print('✅ Mond tracking: lower face region | MAX 1 GEZICHT')

# ============================================
# CONFIG
# ============================================
CONF_THRESHOLD = 0.70
YAWN_THRESHOLD = 0.62
CLOSED_ALARM_SECONDS = 2.0
INFERENCE_SKIP_FRAMES = 2
MIN_FACE_SIZE = (60, 60)
MAX_FACES = 1

# ============================================
# STATE
# ============================================
blink_count = 0
yawn_count = 0
eyes_closed_since = None
was_closed_prev = False
was_yawn_prev = False
alarm_active = False
last_alarm = 0.0

cached_eyes_closed = False
cached_is_yawn = False
cached_eyes_conf = 0.0
cached_yawn_conf = 0.0

fps = 0.0
fps_n = 0
fps_t0 = time.time()

# ============================================
# AUDIO
# ============================================
def play_loud_alarm():
    if winsound:
        for _ in range(10):
            winsound.Beep(1200, 300)
            time.sleep(0.1)
    else:
        for _ in range(15):
            print("\a")
            time.sleep(0.2)

def start_loud_alarm():
    threading.Thread(target=play_loud_alarm, daemon=True).start()

# ============================================
# CAMERA
# ============================================
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1000)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 750)
cap.set(cv2.CAP_PROP_FPS, 30)

print("🎥 1 GEZICHT + YAWN COUNTS | q=quit")

inference_frame_count = 0

# ============================================
# MAIN LOOP
# ============================================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    fps_n += 1
    if time.time() - fps_t0 >= 1.0:
        fps = fps_n / (time.time() - fps_t0)
        fps_t0 = time.time()
        fps_n = 0

    now = time.time()
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.2,
        minNeighbors=6,
        minSize=MIN_FACE_SIZE,
        flags=cv2.CASCADE_SCALE_IMAGE
    )

    if len(faces) > 0:
        areas = [w * h for x, y, w, h in faces]
        idx = int(np.argmax(areas))
        faces = np.array([faces[idx]])

    inference_frame_count += 1
    do_inference = (inference_frame_count % INFERENCE_SKIP_FRAMES == 0)

    is_closed_now = cached_eyes_closed
    is_yawn_now = cached_is_yawn
    eyes_conf = cached_eyes_conf
    yawn_conf = cached_yawn_conf

    if len(faces) == 1 and do_inference:
        fx, fy, fw, fh = faces[0]

        face_crop = frame[fy:fy+fh, fx:fx+fw]
        if face_crop.size > 0:
            face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)

            mouth_h = int(fh * 0.4)
            mouth_y = fy + int(fh * 0.6)
            mouth_crop = frame[mouth_y:mouth_y+mouth_h, fx:fx+fw]

            if mouth_crop.size > 0:
                mouth_rgb = cv2.cvtColor(mouth_crop, cv2.COLOR_BGR2RGB)
                mouth_pil = Image.fromarray(mouth_rgb).resize((224, 224))

                face_pil = Image.fromarray(face_rgb).resize((224, 224))
                eyes_tensor = eyes_transform(face_pil).unsqueeze(0).to(device)
                yawn_tensor = yawn_transform(mouth_pil).unsqueeze(0).to(device)

                with torch.no_grad():
                    eyes_logits = eyes_model(eyes_tensor)
                    eyes_probs = torch.softmax(eyes_logits, 1)
                    eyes_pred = int(torch.argmax(eyes_probs, 1).item())
                    eyes_conf = float(eyes_probs.max().item())
                    cached_eyes_closed = is_closed_now = (eyes_pred == 0) and (eyes_conf > CONF_THRESHOLD)

                    yawn_logits = yawn_model(yawn_tensor)
                    yawn_probs = torch.softmax(yawn_logits, 1)
                    yawn_pred = int(torch.argmax(yawn_probs, 1).item())
                    yawn_conf = float(yawn_probs.max().item())
                    cached_is_yawn = is_yawn_now = (yawn_pred == 1) and (yawn_conf > YAWN_THRESHOLD)

    if was_closed_prev and not is_closed_now:
        blink_count += 1
    if was_yawn_prev and not is_yawn_now:
        yawn_count += 1

    if is_closed_now:
        if eyes_closed_since is None:
            eyes_closed_since = now
    else:
        eyes_closed_since = None

    was_closed_prev = is_closed_now
    was_yawn_prev = is_yawn_now

    closed_duration = 0.0 if eyes_closed_since is None else now - eyes_closed_since

    # ALARM ENKEL BIJ SLAPEN
    if closed_duration >= CLOSED_ALARM_SECONDS:
        alarm_active = True
        flash = (0, 0, 255) if int(now * 5) % 2 else (255, 255, 255)
        cv2.rectangle(frame, (0, 0), (frame.shape[1], frame.shape[0]), flash, 20)
        cv2.putText(frame, '🚨 WAKE UP! SLAPEN GEDETECTEERD 🚨', (160, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 5)
        if now - last_alarm >= 3.0:
            last_alarm = now
            start_loud_alarm()
    else:
        alarm_active = False

    # DRAW ONLY 1 FACE
    if len(faces) == 1:
        fx, fy, fw, fh = faces[0]

        cv2.rectangle(frame, (fx, fy), (fx + fw, fy + fh), (255, 200, 0), 2)

        mouth_h = int(fh * 0.4)
        mouth_y = fy + int(fh * 0.6)
        cv2.rectangle(frame, (fx, mouth_y), (fx + fw, mouth_y + mouth_h), (0, 255, 0), 3)

        color = (0, 0, 255) if is_closed_now else (0, 255, 0)
        status = 'SLAPEN🔴' if is_closed_now else 'OK✅'
        label = f'{status} | E:{eyes_conf:.1f} Y:{yawn_conf:.1f}'
        cv2.putText(frame, label, (fx, fy - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    # UI
    overlay = frame.copy()
    cv2.rectangle(overlay, (15, 15), (550, 220), (20, 20, 20), -1)
    frame = cv2.addWeighted(overlay, 0.6, frame, 0.4, 0)

    cv2.putText(frame, f'👁️  Blinks: {blink_count}', (25, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    cv2.putText(frame, f'😴 Yawns:  {yawn_count}', (25, 80), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 200, 0), 2)
    cv2.putText(frame, f'Eyes closed: {closed_duration:.1f}s', (25, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    cv2.putText(frame, f'Yawn conf: {yawn_conf:.1f}', (25, 135), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    cv2.putText(frame, f'FPS: {fps:.1f} | 1 GEZICHT', (25, 160), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (150, 150, 150), 2)

    cv2.putText(frame, 'q=quit | Groen=Mond | MAX 1 GEZICHT',
                (25, frame.shape[0] - 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (180, 180, 180), 2)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

    cv2.imshow('Fatigue Monitor (1 GEZICHT + MOND COUNTS)', frame)

cap.release()
cv2.destroyAllWindows()
print("✅ MAX 1 GEZICHT - geen glitches meer!")

Imports geladen!
🚀 MPS!
✅ Eyes OK
✅ Yawn OK
✅ Mond tracking: lower face region | MAX 1 GEZICHT
🎥 1 GEZICHT + YAWN COUNTS | q=quit
✅ MAX 1 GEZICHT - geen glitches meer!
